![Banner](../images/09_03_banner_rs_rader.png)


# Sentinel-1 SAR

This notebook introduces Sentinel-1, the C-band radar mission of the Copernicus programme: the satellites and their status, acquisition modes, product levels, polarisations and the main applications of its free, all-weather imagery.


## Overview

Built and operated by the European Space Agency (ESA), it is an active sensor: it supplies its own illumination and works through clouds, smoke, and darkness. It monitors land, ocean, and ice day and night. All data are free and open.


## Core concepts

### Learning objectives

1. State which Sentinel-1 satellites are operational, which have failed or retired, and how the 6-day revisit was restored in 2026.
2. Describe the dawn-dusk orbit, the 12-day repeat cycle, and the 6-day revisit when two satellites are phased.
3. Give the C-band frequency, wavelength, polarisation options, antenna size, and the default IW land mode.
4. Compare IW, EW, Stripmap, and Wave mode by swath, resolution, and main use, including TOPSAR subswaths and bursts.
5. Choose among Level-0, SLC, GRD, OCN, and analysis-ready products for interferometry, backscatter mapping, or ocean variables.
6. Interpret VV and VH backscatter from roughness, moisture, structure, and incidence angle, and recognise speckle, foreshortening, layover, and radar shadow.
7. Name the archives that distribute Sentinel-1 and the main software used to search and process it.

### Contents

1. Constellation history and current status
2. Orbit Details
3. Sentinel-1 SAR Instrument (Exokain)
4. Acquisition Modes
5. Product Levels
6. Backscatter Interpretation
7. SAR-specific effects
8. Applications
9. Data access
10. Software


## Constellation history and current status

| Satellite | Launch | Status |
|---|---|---|
| Sentinel-1A | 3 April 2014 | Retired 29 June 2026 |
| Sentinel-1B | 25 April 2016 | Failed December 2021 (power system anomaly). Mission declared over in 2022 |
| Sentinel-1C | 5 December 2024 (Vega-C) | Operational |
| Sentinel-1D | November 2025 (Ariane 6) | Operational |

From 2022 to 2024, only Sentinel-1A was flying, so revisit dropped to 12 days. That period has a coverage gap that matters for time series work.

ESA opened Sentinel-1D data to users on 17 April 2026, initially on an orbit following Sentinel-1C's ground track with a one-day offset. Sentinel-1D finished commissioning and became fully operational on 1 May 2026. Sentinel-1C was then moved between June 9 and 23, 2026 to set up a six-day revisit with Sentinel-1D, and Sentinel-1A operations ended on June 29, 2026. The constellation is now Sentinel-1C plus Sentinel-1D.

**Note for time series users:** the new C/D orbit configuration shifts the 6-day revisit pattern by one day compared with the old A/B timeline. Acquisition date logic in existing pipelines may need adjusting.

Both 1C and 1D also carry an Automatic Identification System (AIS) receiver that picks up signals from ships in coastal waters and the Arctic. This makes it possible to match radar ship detections with AIS records.


## Orbit Details

The Sentinel-1 satellites follow a near-polar, sun-synchronous, dawn-dusk orbit. This means the satellites travel almost over the poles and cross the equator at the same local solar times on each pass, generally around dawn and dusk. This configuration ensures stable lighting conditions for optical missions, but with radar, it also means consistent geometry between acquisitions.

- **Altitude:** The orbit has a mean altitude of about 693 km above Earth's surface, and an inclination of approximately 98.18°, allowing the satellites to cover nearly the entire Earth's surface.

- **Repeat Cycle:** Each Sentinel-1 satellite completes 175 orbits within a 12-day repeat cycle, meaning it will observe the same location on the ground every 12 days. When two satellites are operational and phased correctly (e.g., Sentinel-1C and 1D), the revisit interval at the equator reduces to 6 days. At higher latitudes (closer to the poles), orbit paths overlap more, so revisit times are even shorter.

- **Viewing Geometry:** The satellites acquire imagery on both ascending (moving from south to north, typically in the evening) and descending (north to south, usually in the morning) passes. This offers two different viewing angles over a location, which can be helpful for detecting changes or interpreting specific scattering effects in radar imagery.


## Sentinel-1 SAR Instrument (Exokain)

| Parameter          | Value                                                     |
|--------------------|-----------------------------------------------------------|
| Frequency Band     | C-band                                                    |
| Center Frequency   | 5.405 GHz (corresponds to a wavelength of ~5.55 cm)       |
| Wavelength         | Approximately 5.55 cm                                     |
| Polarization       | Single (HH or VV) or dual (HH+HV or VV+VH), selectable    |
| Antenna            | 12.3 m × 0.82 m active phased array (electronic steering) |
| Default Land Mode  | Interferometric Wide Swath (IW) mode, VV+VH polarization  |

**Scattering Characteristics at C-band:**

- The C-band radar wavelength (~5.5 cm) is sensitive to surface features that are similar in size, namely, leaves, small branches, crop canopies, and uppermost few centimeters of soil.
- C-band penetrates vegetation and soil less deeply than L-band, so it mainly reveals the structure and moisture of the canopy top and immediate ground surface.
- The active phased array design allows for beam steering and multiple imaging modes.

*Exokain*: For most Earth observation applications over land, Sentinel-1 operates in Interferometric Wide (IW) mode with dual polarization (VV+VH). This configuration enhances sensitivity to both surface and volume scattering, which aids in separating signals from soil, water, crops, and built environments, and supports interferometric applications.


## Acquisition Modes

Sentinel-1 is equipped with several imaging modes, each optimized for different observation needs. The main acquisition modes are:

| Mode                              | Swath Width          | Spatial Resolution (range × azimuth) | Main Use Cases                                |
|------------------------------------|----------------------|--------------------------------------|-----------------------------------------------|
| Interferometric Wide swath (IW)    | 250 km               | 5 × 20 m                             | Standard mode over land; interferometry (InSAR), monitoring agriculture, detecting floods |
| Extra Wide swath (EW)              | 400 km               | 20 × 40 m                            | Sea ice mapping, Arctic and Antarctic regions, maritime surveillance                    |
| Stripmap (SM)                      | 80 km                | 5 × 5 m                              | High-resolution imaging of small areas, islands, and during emergencies                 |
| Wave (WV)                          | 20 × 20 km "vignettes"| 5 × 5 m                              | Measuring ocean wave spectra in the open sea                                          |


- **Interferometric Wide swath (IW):** The default mode for land applications. IW uses the TOPSAR technique (Terrain Observation with Progressive Scans SAR), where the synthetic aperture radar beam is electronically steered both in the azimuth (flight direction) and in range (across-track) directions. This results in a uniform image quality across the swath. The IW mode is divided into 3 subswaths, each covering part of the total swath width, and each subswath is further split into small segments called "bursts." The incidence angle for IW mode ranges from about 29° to 46°, this determines how steeply the radar looks at the ground.

- **Extra Wide swath (EW):** Used mainly for very broad-area coverage like monitoring sea ice and large maritime regions. EW also uses TOPSAR, providing 5 subswaths and a wider coverage at lower resolution.

- **Stripmap (SM):** Provides high-resolution images but over a much narrower area. SM is used when detailed information of a localized region is needed, such as for mapping during disasters.

- **Wave mode (WV):** Instead of continuous stripes, Wave mode acquires small "vignettes" (20 km × 20 km images) at regular intervals along the satellite's ground track. These are used primarily to study ocean surface wave patterns globally.

**Summary:**  
IW and EW modes both use the TOPSAR technology to ensure consistent coverage and radiometric performance across wide swaths. Subswaths and bursts allow the satellite to store and process data efficiently and ensure compatibility with applications like interferometry.


## Product Levels

Sentinel-1 products are available at several levels of processing, with each level designed for specific types of analysis and user requirements:

**Level-0:**  
- These are the unprocessed, original radar echoes collected by the satellite, stored in a compressed format.  
- Data are still in raw instrument units and have not undergone any calibration, geometric correction, or noise removal.  
- Intended only for advanced users or engineers, as specialized knowledge and tools are needed to interpret or process them.

**Level-1 SLC (Single Look Complex):**  
- This product provides data in "slant range" geometry, which preserves the original radar viewing geometry (before correction to ground range).  
- Both amplitude (strength) and phase (timing information) for each radar return are retained as complex values.  
- The presence of phase information is critical for applications like interferometry (InSAR, which measures ground movement using phase differences between repeat images) and polarimetric analysis (studying different polarization channels for target characterization).  
- For the Interferometric Wide Swath (IW) mode, SLC data is organized by "subswaths" (spatial sections across the track) and further split into bursts for efficient handling and to align with acquisition strategy.

**Level-1 GRD (Ground Range Detected):**  
- In these products, the radar data have been calibrated, multilooked (averaged to reduce noise), and geometrically corrected so the image coordinates correspond to the Earth’s surface as projected onto an ellipsoid (approximate globe shape).  
- Only the amplitude (intensity) information is kept; all phase information is lost, meaning these images aren’t suitable for interferometric analysis.  
- Multiple resolutions are available: Full Resolution (FR), High Resolution (HR), and Medium Resolution (MR).  
- For IW mode, the High Resolution GRD (GRDH) product has a pixel spacing of 10 meters, with an effective spatial resolution of roughly 20 × 22 meters, making it well-suited for monitoring land surfaces and water bodies.  
- Level-1 GRD products are the standard for mapping and backscatter time-series analysis, and are compatible with most open analysis workflows.

**Level-2 OCN (Ocean):**  
- These products are processed specifically for oceanographic applications.  
- They contain geophysical variables such as ocean surface wind fields, swell wave spectra (wave energy distribution), and surface radial velocity.  
- Such products support marine forecasts, sea state monitoring, and other ocean applications.

**Analysis Ready Data (ARD):**  
- These are datasets that have undergone additional processing steps (beyond standard Level-1) to be easier and faster to use in downstream analysis. ARD products are standardized to make them interoperable and minimize the need for further user preprocessing.  
- **Copernicus Sentinel-1 Normalized Radar Backscatter:** Follows CEOS ARD standards, making backscatter data directly usable for time series, change detection, and other applications.  
- **NASA JPL OPERA products:** Targeted for North America, these come in several forms:  
-- **RTC-S1:** Radiometrically Terrain Corrected backscatter, with terrain effects removed and data georeferenced to the surface.  
-- **CSLC-S1:** Coregistered Single Look Complex products, designed for direct use in time-series InSAR applications.  
-- **DISP-S1:** Surface displacement time series, giving geocoded maps of ground deformation over time, derived using InSAR.

In summary, the product level you select depends on your required analysis:  
- For interferometry, ground motion mapping, or polarimetry, use Level-1 SLC or CSLC-S1.  
- For surface feature or land/water mapping, use Level-1 GRD, RTC-S1, or ARD backscatter.
- For marine applications, use Level-2 OCN.


## Backscatter Interpretation

In radar remote sensing, the key measurement is *backscatter*, which quantifies how much of the radar signal is reflected back to the sensor from the Earth's surface. This measurement is usually expressed as sigma nought (σ⁰) or gamma nought (γ⁰), which represent the normalized radar cross section, and is often reported in decibels (dB) for easier interpretation.

The intensity of the backscatter received by the sensor is influenced by several factors:

- **Surface roughness:** Smooth surfaces, such as calm water bodies, tend to reflect radar waves away from the sensor (specular reflection), resulting in a weaker signal and appearing dark in the radar image. In contrast, rough surfaces scatter the radar waves in many directions, including back toward the sensor, causing them to appear brighter.

- **Dielectric constant (moisture content):** The ability of a material to transmit electrical fields affects how it interacts with radar signals. Materials with higher moisture content, such as wet soils, saturated vegetation, or flooded areas, generally have higher dielectric constants and return stronger backscatter signals.

- **Geometry and structure:** The physical structure of objects dramatically affects the radar return. For example, buildings or other man-made objects often create "double-bounce" reflections, where the signal bounces between the ground and a vertical structure (like a building wall), returning a particularly strong signal to the sensor and causing urban areas to appear very bright. Vegetation canopies, due to their volume and internal layering, can create volume scattering, especially in the cross-polarized (VH) channel where the sensor transmits and receives in different polarizations.

- **Incidence angle:** This is the angle between the radar beam and a perpendicular to the surface (nadir). As the incidence angle increases (i.e., the beam strikes the surface more obliquely, farther from the center of the image swath), backscatter levels typically decrease. Therefore, data are often "normalized" to correct for differences in incidence angle across the swath to make backscatter values comparable.

**Typical Backscatter Patterns by Land Cover Type:**

| Surface Type        | VV Polarization       | VH Polarization        |
|--------------------|----------------------|------------------------|
| Open water         | Very low             | Very low               |
| Bare smooth soil   | Low to moderate      | Low                    |
| Crops and forest   | Moderate             | Moderate to high       |
| Urban              | Very high            | High                   |

In this table:
- **VV (Vertical transmit, Vertical receive):** Sensitive to surface roughness and structure, and commonly used in standard analyses.
- **VH (Vertical transmit, Horizontal receive):** Often emphasizes scattering from randomly oriented or volumetric structures (like vegetation canopies), making it particularly useful for monitoring volume scatterers such as crops and forests.

**Ratios and Indices:**

- The **VH/VV backscatter ratio** can highlight differences in land cover, areas with higher ratios often indicate dense or complex vegetation structures.
- The **Radar Vegetation Index (RVI)** is a derived metric that quantifies the amount of volume scattering and is widely used for crop identification, vegetation status monitoring, and assessing plant health or biomass.

Understanding these details enables effective interpretation of radar images, such as distinguishing between water, soil, vegetation, and built environments, and monitoring changes in surface conditions over time.


## SAR-specific effects

Synthetic Aperture Radar (SAR) imagery exhibits several unique effects and artifacts due to the way radar signals interact with the Earth's surface and the geometry of imaging. Understanding these effects is essential for accurate interpretation and analysis:

- **Speckle:** 
  - *What it is:* A granular, salt-and-pepper appearance resulting from coherent interference between radar returns from multiple scattering elements within a single resolution cell.
  - *Why it matters:* Speckle is not random noise but is instead inherent to the SAR imaging process, potentially obscuring fine details and making interpretation challenging.
  - *How it's mitigated:* Processing techniques such as Lee, Refined Lee, or Gamma-MAP filtering, as well as multitemporal (time series) averaging, are commonly used to reduce speckle while preserving structural detail.

- **Foreshortening:**
  - *What it is:* When slopes face toward the radar sensor, the distance between radar pulses is shorter than on flat ground, causing these slopes to appear artificially compressed ("foreshortened") in the image.
  - *Why it matters:* Steep terrain facing the sensor appears squashed together, sometimes making interpretation or measurement of such areas difficult.

- **Layover:**
  - *What it is:* In mountainous regions, radar signals may reach the top of a slope before the base (since the top is closer to the sensor). As a result, the top is imaged earlier than lower points, so the normal geometry is inverted (“laid over” the base).
  - *Why it matters:* Features appear in unnatural positions, with mountain or building tops sometimes overlaid on top of their slopes or bases in the image.

- **Radar shadow:**
  - *What it is:* Areas that slope away from the radar sensor do not reflect any return signal back, creating dark, shadowed regions in the image.
  - *Why it matters:* These shadows contain little or no radar information and can hide features located within them, acting similarly to optical shadows but resulting from the imaging geometry.

- **Border and thermal noise:**
  - *What it is:* Artifacts at the image borders (border noise) or from the sensor’s electronics (thermal noise). These effects are especially noticeable in cross-polarized images, or over dark, low-backscatter surfaces such as open water.
  - *Why it matters:* They can appear as low-level artifacts or brightness variations, potentially misleading interpretation unless corrected during preprocessing.


## Applications

- **Agriculture:** crop type mapping, rice area and phenology (flooding and transplanting show up clearly in VH time series), crop growth monitoring, yield modeling
- **Soil moisture:** surface moisture retrieval, often fused with optical NDVI
- **Floods and water:** near-real-time flood extent, wetland dynamics. Copernicus Global Flood Monitoring uses it operationally
- **Forests:** deforestation and disturbance detection (for example, tropical forest alerts)
- **Geohazards:** earthquakes, volcanic deformation, landslides, land subsidence from groundwater pumping
- **Infrastructure:** settlement of buildings, dams, and transport networks
- **Ocean and ice:** ship detection, oil spills, sea ice classification, iceberg tracking, wind and wave fields
- **Urban:** settlement mapping and urban growth


## Data access

| Platform | What you get |
|---|---|
| Copernicus Data Space Ecosystem | Official archive. All levels, APIs, openEO, Sentinel Hub |
| ASF DAAC (Vertex, `asf_search`) | Full archive. HyP3 on-demand RTC and InSAR processing. OPERA products |
| Google Earth Engine | `COPERNICUS/S1_GRD`, already calibrated, terrain corrected, and in dB |
| Microsoft Planetary Computer | `sentinel-1-grd` and `sentinel-1-rtc` STAC collections |
| AWS Open Data | GRD and SLC bursts |


## Software

- **ESA SNAP** with `esa_snappy` (GUI and graph processing)
- **ISCE2 / ISCE3**, **GMTSAR**, **Dolphin** (InSAR processing)
- **MintPy** (InSAR time series)
- **pyroSAR**, **sarsen**, **s1-reader** (Python processing)
- **asf_search**, **pystac-client**, **stackstac**, **xarray**, **rioxarray** (search and analysis)
- **earthengine-api / geemap** (cloud processing)
- **GeoLattice:** GeoLattice is a Python library designed for geospatial data analysis and processing, with tools tailored for working with SAR (Synthetic Aperture Radar) datasets such as Sentinel-1. It provides user-friendly functions for reading, visualizing, and analyzing radar data, making it easier to perform common SAR processing tasks like data conversion, filtering, and feature extraction within a Python workflow.


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.